# Python for Bioinformatics — Part III Solutions
## Modules 9–12

Reference solutions with commentary. Run from the folder containing `data/`. Plots will display inline in a notebook.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
print("imports ready")

## Module 9 — NumPy

**9A.1**

In [ ]:
lengths = np.array([150, 148, 152, 149, 151, 75])
print(lengths.shape, lengths.dtype)
print("mean", lengths.mean(), "min", lengths.min(), "max", lengths.max())

**9A.2**

In [ ]:
starts = np.arange(0, 31, 3)
print(starts, "len", len(starts))
m = np.arange(12).reshape(3, 4)
print(m, m.shape)

**9B.1** — boolean masking.

In [ ]:
quals = np.array([40, 38, 12, 40, 8, 39, 25])
print(quals[quals >= 30])
print("count <20:", (quals < 20).sum())
q2 = quals.copy()
q2[q2 < 20] = 0
print(q2)

**9B.2** — Phred → error probability.

In [ ]:
quals = np.array([40, 30, 20, 10])
err = 10 ** (-quals / 10)
print(np.round(err, 4))   # [0.0001 0.001  0.01   0.1   ]

**9B.3** — fold change + flag with np.select.

In [ ]:
control = np.array([120.5, 45.0, 88.0, 33.2])
treated = np.array([97.5, 204.0, 85.3, 138.0])
genes = ["rpoB", "katG", "gyrA", "inhA"]
log2fc = np.log2(treated / control)
flags = np.select([log2fc > 1, log2fc < -1], ["up", "down"], default="flat")
for g, fc, fl in zip(genes, log2fc, flags):
    print(f"{g}: {fc:+.2f} {fl}")

**9C.1**

In [ ]:
m = np.array([[10, 20, 30], [5, 15, 25], [50, 40, 30]])
print("mean per gene (axis=1):", m.mean(axis=1))
print("max per sample (axis=0):", m.max(axis=0))

**Module 9 challenge** — vectorised expression analysis.

In [ ]:
genes = np.array(["rpoB", "katG", "gyrA", "inhA"])
expr = np.array([[120.5, 118.2, 95.1, 99.8],
                 [ 45.0,  47.3, 210.4, 198.7],
                 [ 88.0,  91.2,  84.5,  86.1],
                 [ 33.2,  31.9, 140.8, 135.5]])
mean_control = expr[:, 0:2].mean(axis=1)   # first two columns
mean_treated = expr[:, 2:4].mean(axis=1)   # last two columns
log2fc = np.log2(mean_treated / mean_control)
flags = np.select([log2fc > 1, log2fc < -1], ["up", "down"], default="flat")
for g, fc, fl in zip(genes, log2fc, flags):
    print(f"{g}: log2FC {fc:+.2f}  ->  {fl}")

## Module 10 — pandas

**10A.1**

In [ ]:
df = pd.DataFrame({"gene": ["rpoB","katG","gyrA","inhA"],
                   "length": [3534, 2223, 2517, 810]})
print(df)
print("shape", df.shape, "| mean length", df["length"].mean())

**10A.2**

In [ ]:
variants = pd.read_csv("data/variants.csv")
print(list(variants.columns))
print(variants.dtypes)
print(variants.describe())

**10B.1**

In [ ]:
variants = pd.read_csv("data/variants.csv")
print(variants[["gene", "position"]])
print(variants.iloc[0:3])

**10B.2** — combined filters.

In [ ]:
variants = pd.read_csv("data/variants.csv")
print(variants[(variants["gene"] == "gyrA") & (variants["quality"] > 40)])
print(variants[(variants["quality"] < 40) | (variants["gene"] == "pncA")])

**10B.3** — new columns.

In [ ]:
variants = pd.read_csv("data/variants.csv")
variants["error_prob"] = 10 ** (-variants["quality"] / 10)
variants["pass_qc"] = variants["quality"] >= 40
print(variants)

**10C.1**

In [ ]:
expr = pd.read_csv("data/expression.csv")
print(expr.groupby("gene")["tpm"].mean())

**10C.2**

In [ ]:
expr = pd.read_csv("data/expression.csv")
print(expr.groupby(["gene","condition"])["tpm"].mean())
print(expr.groupby("condition")["tpm"].agg(["mean","std"]))

**Module 10 challenge** — mini expression pipeline.

In [ ]:
expr = pd.read_csv("data/expression.csv")
means = expr.groupby(["gene","condition"])["tpm"].mean().reset_index()
wide = means.pivot_table(index="gene", columns="condition", values="tpm")
wide["log2fc"] = np.log2(wide["treated"] / wide["control"])
wide = wide.sort_values("log2fc", ascending=False)
wide.to_csv("expression_summary.csv")
print(wide)

## Module 11 — Visualisation

**11A.1** — bar chart.

In [ ]:
genes = ["rpoB","katG","gyrA","inhA"]
lengths = [3534, 2223, 2517, 810]
fig, ax = plt.subplots(figsize=(6,4))
ax.bar(genes, lengths, color="darkorange")
ax.set_xlabel("Gene"); ax.set_ylabel("Length (bp)"); ax.set_title("Gene lengths")
fig.tight_layout()
plt.show()

**11A.2** — histogram, saved to PNG.

In [ ]:
mean_quals = [40.0, 38.5, 7.6, 40.0]
fig, ax = plt.subplots(figsize=(6,4))
ax.hist(mean_quals, bins=8, color="seagreen", edgecolor="black")
ax.set_xlabel("Mean read quality (Phred)"); ax.set_ylabel("Count")
ax.set_title("Per-read quality distribution")
fig.tight_layout()
fig.savefig("quality_hist.png", dpi=120)
plt.show()
print("saved quality_hist.png")

**11B.1** — seaborn grouped bar.

In [ ]:
expr = pd.read_csv("data/expression.csv")
ax = sns.barplot(data=expr, x="gene", y="tpm", hue="condition")
ax.set_title("Mean TPM per gene by condition")
plt.show()

**11B.2** — boxplot, customised via returned ax.

In [ ]:
expr = pd.read_csv("data/expression.csv")
ax = sns.boxplot(data=expr, x="gene", y="tpm")
ax.set_ylabel("TPM")
ax.set_title("Expression distribution per gene")
plt.show()

**Module 11 challenge** — two-panel figure.

In [ ]:
expr = pd.read_csv("data/expression.csv")
gene_means = expr.groupby("gene")["tpm"].mean()
cond_means = expr.groupby(["gene","condition"])["tpm"].mean().unstack()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11,4))
ax1.bar(gene_means.index, gene_means.values, color="steelblue")
ax1.set_title("Mean TPM per gene"); ax1.set_ylabel("TPM")

ax2.scatter(cond_means["control"], cond_means["treated"], s=60)
lims = [0, max(cond_means.max()) * 1.1]
ax2.plot(lims, lims, "k--", alpha=0.5)   # y = x reference
ax2.set_xlabel("control TPM"); ax2.set_ylabel("treated TPM")
ax2.set_title("Control vs treated")
for gene in cond_means.index:
    ax2.annotate(gene, (cond_means.loc[gene,"control"], cond_means.loc[gene,"treated"]))

fig.suptitle("Expression overview")
fig.tight_layout()
fig.savefig("expression_panels.png", dpi=120)
plt.show()
print("saved expression_panels.png")

## Module 12 — Biopython

**12A.1**

In [ ]:
from Bio.Seq import Seq
dna = Seq("ATGCGTACGTTAGC")
print("len", len(dna))
print("rev-comp", dna.reverse_complement())
print("RNA", dna.transcribe())

**12A.2**

In [ ]:
cds = Seq("ATGAAACCCGGGTGGTAA")
print("standard:", cds.translate())             # ...* at stop
print("bacterial, to_stop:", cds.translate(table=11, to_stop=True))

**12B.1**

In [ ]:
from Bio import SeqIO
for record in SeqIO.parse("data/sample.fasta", "fasta"):
    print(record.id, str(record.seq[:10]))

**12B.2** — Biopython decodes Phred scores for you.

In [ ]:
for record in SeqIO.parse("data/sample.fastq", "fastq"):
    q = record.letter_annotations["phred_quality"]
    print(record.id, "mean Q:", round(sum(q)/len(q), 1))

**Part III capstone** — end-to-end mini-pipeline.

In [ ]:
from Bio import SeqIO

rows = []
for record in SeqIO.parse("data/sample.fasta", "fasta"):
    seq = record.seq
    gc = 100 * (seq.count("G") + seq.count("C")) / len(seq)
    protein = seq.translate(table=11, to_stop=True)
    rows.append({
        "id": record.id,
        "length": len(seq),
        "gc_percent": round(gc, 1),
        "protein_length": len(protein),
    })

df = pd.DataFrame(rows)
print(df)
print(f"\nMean length: {df['length'].mean():.0f} bp")
print(f"Mean GC: {df['gc_percent'].mean():.1f}%")

fig, ax = plt.subplots(figsize=(6,4))
ax.bar(df["id"], df["gc_percent"], color="mediumpurple")
ax.set_xlabel("Gene"); ax.set_ylabel("GC %"); ax.set_title("GC content per sequence")
fig.tight_layout()
fig.savefig("pipeline_gc.png", dpi=120)
plt.show()
print("saved pipeline_gc.png")

---
*Part III complete.* You can load, manipulate, summarise, visualise, and handle real sequence data with the field's standard tools — and you understand what each does because you built simpler versions by hand. Part IV: idiomatic Python (comprehensions, generators, decorators), Git/GitHub, and your capstone tool.